# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I am working on **Lane 2: Content Refresh / Opportunity Scoring**.

The task type is **binary classification**.

The question I am answering is: *"Will this page's search performance decline over the next period?"*

This is a classification problem because the answer is a hard yes or no — a page is either declining or it is not. I am not trying to rank pages against each other (ranking), group them into unknown categories (clustering), or produce a continuous score with no floor or ceiling (regression). I am predicting a real, observed outcome that I can see in the data after the fact, which is what makes it a classification task and not a guess.

The output supports a specific editorial decision: which pages should an editor prioritise for a content refresh? That makes it a decision-support tool, not just a prediction.

In [1]:
# Section 1 — no code needed here; the framing is in the markdown above.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target I am predicting is whether a page is **declining** in search performance.

The dataset has a column called `trend_direction` with values: `down`, `stable`, `up`, `new`, and `flat`. I derive the binary label from this: `trend_direction == 'down'` → label 1 (declining), everything else → label 0 (not declining).

This label is **observed in the data** — it comes from a measured percentage change in impressions over a 90-day window (`trend_pct`), not a rule I invented.

**Critical rule I am following:** because the label is derived from `trend_direction` and `trend_pct`, those two columns are **never features**. If I used them to predict the label they generated, the model would just learn its own definition back — that is called target leakage, and it produces fake perfect results that break on real data. Every other column — position, CTR, engagement, content age, word count, content type — is a legitimate feature.

In [2]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Derive the binary label: declining = 1, everything else = 0
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Show label distribution
label_counts = df['is_declining'].value_counts()
label_pct = df['is_declining'].value_counts(normalize=True) * 100

print('Target column: is_declining  (derived from trend_direction == "down")')
print('-----------------------------------------------------------------------')
print(f'  Not declining (0): {label_counts[0]:,} rows ({label_pct[0]:.1f}%)')
print(f'  Declining     (1): {label_counts[1]:,} rows ({label_pct[1]:.1f}%)')
print(f'  Total:             {len(df):,} rows')
print()
print('trend_direction breakdown (for reference):')
print(df['trend_direction'].value_counts().to_string())
print()
print('Note: trend_direction and trend_pct are excluded from features — they created this label.')

Target column: is_declining  (derived from trend_direction == "down")
-----------------------------------------------------------------------
  Not declining (0): 13,738 rows (45.8%)
  Declining     (1): 16,262 rows (54.2%)
  Total:             30,000 rows

trend_direction breakdown (for reference):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Note: trend_direction and trend_pct are excluded from features — they created this label.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary metric is **ROC-AUC** (Area Under the Receiver Operating Characteristic curve).

ROC-AUC measures how well the model separates declining pages from stable ones across all possible decision thresholds. A score of 0.5 means the model is no better than a coin flip. A score of 1.0 means perfect separation. I can defend this metric because:

1. The label is moderately imbalanced — about 54% of pages are declining and 46% are not. ROC-AUC handles imbalanced classes better than raw accuracy.
2. I do not need to pick a single threshold upfront. Editors can tune how aggressive or conservative they want the tool to be.

**What 'good' looks like:** a ROC-AUC above 0.70 would mean the model meaningfully outperforms random prioritisation. My baseline to beat is a simple rule (e.g. flag all pages with a position worse than 10) — I compute that baseline in the code below.

**Secondary metric — Precision at top-K:** Editors have a fixed weekly bandwidth. They can realistically refresh maybe 20–50 pages per week. So I also care about how many of the top-ranked pages are genuinely declining. This is not my optimisation target, but it is the metric that translates to real editorial value.

In [3]:
from sklearn.metrics import roc_auc_score

# Handle avg_position: 0 means missing data, not rank zero
df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)

# Naive rule baseline: flag pages with avg_position worse than 10
# Only compute where position data is available
mask = df['avg_position_clean'].notna()
rule_based_pred = (df.loc[mask, 'avg_position_clean'] > 10).astype(int)
rule_auc = roc_auc_score(df.loc[mask, 'is_declining'], rule_based_pred)

# Majority-class baseline: predict everything as declining
majority_pred = np.ones(len(df))
majority_auc = roc_auc_score(df['is_declining'], majority_pred)

print('Baseline metrics (things my model needs to beat):')
print('--------------------------------------------------')
print(f'  Majority class baseline AUC:                      {majority_auc:.3f}')
print(f'  Naive position rule (avg_position > 10) AUC:      {rule_auc:.3f}')
print()
print('Any model with AUC clearly above the position rule baseline')
print('is already more useful than a simple threshold rule.')

Baseline metrics (things my model needs to beat):
--------------------------------------------------
  Majority class baseline AUC:                      0.500
  Naive position rule (avg_position > 10) AUC:      0.501

Any model with AUC clearly above the position rule baseline
is already more useful than a simple threshold rule.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page, for one client, measured over a trailing 90-day window.**

Every row represents a single piece of content (an article, a landing page, a product page) belonging to a specific client. The metrics in that row — impressions, clicks, average position, CTR, engagement rate — all summarise that page's performance over the last 90 days.

The columns I care most about for this lane:
- **IDs:** `content_id`, `client_id` (for grouping only — never features)
- **Search signals:** `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`
- **Engagement signals:** `engagement_rate`, `scroll_rate`
- **Content metadata:** `content_type`, `word_count`, `days_since_last_update`
- **Target:** `is_declining` (derived above — not in the raw file)

In [4]:
# Columns relevant to Lane 2
lane_cols = [
    'content_id', 'client_id', 'content_type',
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'word_count',
    'days_since_last_update', 'is_declining'
]

lane_df = df[lane_cols].copy()

print(f'Shape: {lane_df.shape[0]:,} rows x {lane_df.shape[1]} columns')
print(f'Clients: {lane_df["client_id"].nunique()}')
print(f'Content types: {lane_df["content_type"].unique().tolist()}')
print()
print('Sample rows (one row = one content page for one client, 90-day window):')
print('-----------------------------------------------------------------------')
print(lane_df.sample(5, random_state=42).reset_index(drop=True).to_string())

print()
print('Missing value summary for key columns:')
null_summary = lane_df[['avg_position', 'word_count', 'engagement_rate', 'scroll_rate']].isnull().sum()
print(null_summary.rename('null_count').to_string())
print()
print('Note: avg_position == 0 means missing data (1,205 rows) — not rank zero.')

Shape: 30,000 rows x 12 columns
Clients: 32
Content types: ['keyword article', 'feedly article', 'comparison article']

Sample rows (one row = one content page for one client, 90-day window):
-----------------------------------------------------------------------
             content_id          client_id        content_type  impressions_90d  clicks_90d   ctr  avg_position  engagement_rate  scroll_rate  word_count  days_since_last_update  is_declining
0  content_9824710082d8  client_3fdba35f04     keyword article              283           0  0.00          21.3             0.00         0.00      1397.0                     104             0
1  content_3efa3a7c46bb  client_f74efabef1     keyword article             8878           8  0.09           8.8             4.35         4.35      3188.0                      20             0
2  content_575dc8a2ab0f  client_25fc0e7096      feedly article                3           0  0.00           0.0             0.00        20.83      3381.0       

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single rule cannot capture this problem. Here is why:

**The signals interact in ways a threshold cannot handle.** A page with a low CTR might still be perfectly stable if it sits in position 1 for a low-competition keyword. A page with high impressions might still be declining if its engagement rate is dropping fast. Decline is driven by the *combination* of position, CTR, content freshness, content type, and engagement — not any one factor on its own.

**The threshold would be different for every content type.** A blog post and a product page behave differently. A rule that correctly flags declining blog posts will flood the queue with false positives for product pages. ML can learn those conditional patterns; a hand-written if-statement cannot without becoming impossibly complex.

**The pattern shifts across clients and niches.** What counts as a bad position for one client in one niche is fine for another. A fixed rule either gets too generic (misses real declines) or too specific (flags everything). ML learns from the actual outcomes in the data.

**In short:** if I could write a reliable rule, I would. The fact that I cannot — and the fact that the data contains over 40 potentially relevant signals — is exactly why a trained model earns its place here. The output supports a clear action: an editor looks at the ranked list and decides which pages to refresh first.

In [5]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for script execution
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: CTR distribution by label
for label, group in df.groupby('is_declining'):
    name = 'Declining' if label == 1 else 'Not declining'
    axes[0].hist(group['ctr'].clip(upper=5), bins=40, alpha=0.55, label=name, density=True)
axes[0].set_title('CTR by label\n(overlap = no single CTR threshold works)')
axes[0].set_xlabel('CTR (x100 scale)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Plot 2: avg_position distribution by label (excluding 0 = missing)
pos_df = df[df['avg_position'] > 0]
for label, group in pos_df.groupby('is_declining'):
    name = 'Declining' if label == 1 else 'Not declining'
    axes[1].hist(group['avg_position'].clip(upper=50), bins=40, alpha=0.55, label=name, density=True)
axes[1].set_title('avg_position by label\n(overlap = no single position threshold works)')
axes[1].set_xlabel('Average position (lower = better)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
import os
os.makedirs('../../work/outputs', exist_ok=True)
plt.savefig('../../work/outputs/w02_signal_overlap.png', dpi=120, bbox_inches='tight')
print('Saved: work/outputs/w02_signal_overlap.png')
print()
print('Both charts show heavy overlap between declining and not-declining pages.')
print('No single threshold cleanly separates them — ML is the right tool here.')

Saved: work/outputs/w02_signal_overlap.png

Both charts show heavy overlap between declining and not-declining pages.
No single threshold cleanly separates them — ML is the right tool here.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.